imports

In [1]:

import os
import shutil
import tarfile
import torch
import torch.nn as nn
import torch.optim as optim
import boto3
import sagemaker
from sagemaker.pytorch import PyTorch, PyTorchModel 
from dotenv import load_dotenv

load_dotenv()

sagemaker.config INFO - Not applying SDK defaults from location: C:\ProgramData\sagemaker\sagemaker\config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: C:\Users\qyptn\AppData\Local\sagemaker\sagemaker\config.yaml


True

functions

In [2]:
# take inference.py, drowsy_frozen.pt, texting_frozen.pt and zip them to .tgz
def make_tgz(name="model", path="model"):
    shutil.make_archive(
        base_name=name,
        format="gztar",
        root_dir=path
    )

def upload_to_bucket(name="model.tar.gz"):
    print("Uploading to S3...")
    try:
        session = sagemaker.Session()
        try:
            role = sagemaker.get_execution_role()
        except (ValueError, RuntimeError): #if not running on SageMaker
            role = os.getenv("ROLE")
        bucket = session.default_bucket()
        print(f"Bucket: {bucket}")
    except Exception as e:
        print(e)
        exit(1)

    s3_prefix = 'LinModDemo'
    s3_model_path = session.upload_data(path="model.tar.gz", bucket=bucket, key_prefix=s3_prefix)
    print("model_path:", s3_model_path)
    return s3_model_path, session

# sagemaker to make endpoint
def create_endpoint(data, session=None):
    try:
        role = sagemaker.get_execution_role()
    except (ValueError, RuntimeError): #if not running on SageMaker
        print("running using os env role")
        role = os.getenv("ROLE")

    #print("role:", role)
    print("creating pytorchmodel")
    model = PyTorchModel(
    model_data=data,
    role=role,
    framework_version='2.0.0',
    py_version='py310',
    entry_point='inference.py',
    sagemaker_session=session
    )
    print("attempting to deploy")
    predictor = model.deploy(
        initial_instance_count=1,
        instance_type="ml.m5.xlarge",
        endpoint_name="distracted-driver-endpoint"
    )
    print("deployed at:", predictor.endpoint_name)
    return predictor

def cleanup(predictor):
    print("deleting endpoint...")
    predictor.delete_endpoint()

def cleanup_name(endpoint_name, session=None):
    print("deleting endpoint with name:", endpoint_name)
    session.delete_endpoint(endpoint_name)

In [5]:
# upload to default sagemaker bucket
s3_model_path, session = upload_to_bucket()

Uploading to S3...


Couldn't call 'get_role' to get Role ARN from role name andy0102 to get Role path.


Bucket: sagemaker-us-east-1-510497448342
model_path: s3://sagemaker-us-east-1-510497448342/LinModDemo/model.tar.gz


In [4]:
#make_tgz(name="drowsy_model", path="single_model")
make_tgz()

In [34]:
predictor = create_endpoint("s3://sagemaker-us-east-1-510497448342/LinModDemo/model.tar.gz", session)

Couldn't call 'get_role' to get Role ARN from role name andy0102 to get Role path.


running using os env role
creating pytorchmodel
attempting to deploy


KeyboardInterrupt: 

In [25]:
def test_upload_one_model():
    session = sagemaker.Session()
    try:
        role = sagemaker.get_execution_role()
    except (ValueError, RuntimeError): #if not running on SageMaker
        print("running using os env role")
        role = os.getenv("ROLE")

    #print("role:", role)
    print("creating pytorchmodel")
    model = PyTorchModel(
    model_data="s3://distracted-driver-cnn-models/drowsy_model.tar.gz",
    role=role,
    framework_version='2.0.0',
    py_version='py310',
    entry_point='inference.py',
    sagemaker_session=session
)
    print("attempting to deploy")
    predictor = model.deploy(
        initial_instance_count=1,
        instance_type="ml.m5.xlarge",
        endpoint_name="distracted-driver-endpoint"
    )
    print("deployed at:", predictor.endpoint_name)
    return predictor
test_upload_one_model() # upload

Couldn't call 'get_role' to get Role ARN from role name andy0102 to get Role path.


running using os env role
creating pytorchmodel
attempting to deploy


KeyboardInterrupt: 

click below to make tgz

In [ ]:
make_tgz()

In [6]:
# test out our predictor using tst.jpg
with open("tst.jpg", "rb") as f:
    img_bytes = f.read()

response = predictor.predict(
    img_bytes,
    initial_args={"ContentType": "application/octet-stream"}
)

print(response)

NameError: name 'predictor' is not defined

In [ ]:
cleanup(predictor, session) # click me to delete endpoint

NameError: name 'predictor' is not defined